# Visualizacion de las ejecuciones de sysid

Compara, para cada senal (`chirp`, `multisine`, `steps`, `ramp`), la **senal planeada** (`sysid_<senal>.csv`) con la **ejecucion** en el motor (`ejecucion_<senal>.csv`).

Columnas de la ejecucion: `t_cmd_s`/`cmd_deg` (comando enviado), `t_meas_s`/`meas_deg` (angulo medido), `ref_deg` (referencia interpolada en `t_meas_s`) y `error_deg` (`ref - meas`).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DIR = Path('.')
SENALES = ['chirp', 'multisine', 'steps', 'ramp']
plt.rcParams['figure.dpi'] = 100

plan = {s: pd.read_csv(DIR / f'sysid_{s}.csv') for s in SENALES if (DIR / f'sysid_{s}.csv').exists()}
ejec = {s: pd.read_csv(DIR / f'ejecucion_{s}.csv') for s in SENALES if (DIR / f'ejecucion_{s}.csv').exists()}
print({s: len(d) for s, d in ejec.items()})

{'chirp': 1000, 'multisine': 1000, 'steps': 1000, 'ramp': 1000}


## Resumen de metricas

Error RMS y maximo, rango recorrido y retardo estimado por correlacion cruzada entre la referencia y la medicion (positivo = el motor va atrasado).

In [2]:
def retardo(d, max_s=0.5):
    t = d['t_meas_s'].to_numpy(); dt = np.median(np.diff(t))
    ref = d['ref_deg'].to_numpy() - d['ref_deg'].mean(); med = d['meas_deg'].to_numpy() - d['meas_deg'].mean()
    n = int(max_s / dt); lags = np.arange(-n, n + 1)
    c = [np.dot(ref[max(0, -k):len(ref) - max(0, k)], med[max(0, k):len(med) - max(0, -k)]) for k in lags]
    return lags[int(np.argmax(c))] * dt

filas = []
for s, d in ejec.items():
    e = d['error_deg']
    filas.append(dict(senal=s, muestras=len(d), duracion_s=d['t_meas_s'].iloc[-1],
                      rango_ref_deg=d['ref_deg'].max() - d['ref_deg'].min(),
                      rms_err_deg=np.sqrt((e**2).mean()), max_err_deg=e.abs().max(),
                      retardo_ms=1000 * retardo(d),
                      dt_cmd_ms=1000 * d['t_cmd_s'].diff().median(), dt_meas_ms=1000 * d['t_meas_s'].diff().median()))
resumen = pd.DataFrame(filas).set_index('senal').round(3)
resumen

,muestras,duracion_s,rango_ref_deg,rms_err_deg,max_err_deg,retardo_ms,dt_cmd_ms,dt_meas_ms
senal,,,,,,,,
chirp,1000,9.991,29.999,6.806,32.778,100.0,10.0,10.0
multisine,1000,9.991,18.286,2.698,12.215,80.0,10.0,10.0
steps,1000,9.991,10.000,0.847,5.030,60.0,10.0,10.0
ramp,1000,9.991,3.997,0.058,0.442,10.0,10.0,10.0


## Senal planeada vs. ejecutada

Arriba: referencia, comando enviado y angulo medido. Abajo: error de seguimiento.

In [3]:
def graficar(s):
    d = ejec[s]
    fig, (a, b) = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw=dict(height_ratios=[3, 1.3]))
    if s in plan:
        a.plot(plan[s]['time_s'], plan[s]['angle_deg'], color='lightgray', lw=4, label='plan (sysid csv)')
    a.plot(d['t_meas_s'], d['ref_deg'], 'k--', lw=1, label='ref')
    a.plot(d['t_cmd_s'], d['cmd_deg'], color='tab:orange', lw=1, label='comando')
    a.plot(d['t_meas_s'], d['meas_deg'], color='tab:blue', lw=1.3, label='medido')
    a.set_ylabel('angulo [deg]'); a.grid(alpha=0.3); a.legend(ncol=4, fontsize=8)
    a.set_title(f'{s}: RMS = {np.sqrt((d.error_deg**2).mean()):.2f} deg, max = {d.error_deg.abs().max():.2f} deg')
    b.plot(d['t_meas_s'], d['error_deg'], color='tab:red', lw=1)
    b.axhline(0, color='k', lw=0.5); b.set_ylabel('error [deg]'); b.set_xlabel('t [s]'); b.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

for s in ejec:
    graficar(s)

C:\Users\andre\AppData\Local\Temp\ipykernel_28612\2016271978.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Zoom a una ventana de tiempo

Cambia `SENAL`, `T0` y `T1` para ver el detalle.

In [4]:
SENAL, T0, T1 = 'chirp', 0.0, 3.0

d = ejec[SENAL]; m = d['t_meas_s'].between(T0, T1); c = d['t_cmd_s'].between(T0, T1)
plt.figure(figsize=(11, 4))
plt.plot(d.t_meas_s[m], d.ref_deg[m], 'k--', label='ref')
plt.step(d.t_cmd_s[c], d.cmd_deg[c], where='post', color='tab:orange', label='comando')
plt.plot(d.t_meas_s[m], d.meas_deg[m], 'o-', ms=3, color='tab:blue', label='medido')
plt.xlabel('t [s]'); plt.ylabel('angulo [deg]'); plt.grid(alpha=0.3); plt.legend(); plt.title(f'{SENAL} [{T0}, {T1}] s')
plt.tight_layout(); plt.show()

C:\Users\andre\AppData\Local\Temp\ipykernel_28612\3498840561.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Medido vs. referencia

Si el seguimiento fuera perfecto los puntos caerian sobre la diagonal.

In [5]:
fig, axs = plt.subplots(1, len(ejec), figsize=(4 * len(ejec), 4))
for ax, (s, d) in zip(np.atleast_1d(axs), ejec.items()):
    ax.plot(d.ref_deg, d.meas_deg, '.', ms=2)
    lim = [min(d.ref_deg.min(), d.meas_deg.min()), max(d.ref_deg.max(), d.meas_deg.max())]
    ax.plot(lim, lim, 'k--', lw=0.8); ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_title(s); ax.set_xlabel('ref [deg]'); ax.set_ylabel('medido [deg]')
plt.tight_layout(); plt.show()

C:\Users\andre\AppData\Local\Temp\ipykernel_28612\95316138.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Respuesta en frecuencia (chirp y multisine)

Cociente de espectros medido/ref: magnitud (dB) y fase. Solo es fiable donde la referencia tiene energia.

In [6]:
fig, axs = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for j, s in enumerate([x for x in ('chirp', 'multisine') if x in ejec]):
    d = ejec[s]; t = d.t_meas_s.to_numpy(); dt = np.median(np.diff(t))
    r = d.ref_deg.to_numpy() - d.ref_deg.mean(); y = d.meas_deg.to_numpy() - d.meas_deg.mean()
    f = np.fft.rfftfreq(len(r), dt); R = np.fft.rfft(r); Y = np.fft.rfft(y)
    ok = np.abs(R) > 0.05 * np.abs(R).max()
    H = np.where(ok, Y / np.where(ok, R, 1), np.nan)
    axs[0, j].semilogx(f[1:], 20 * np.log10(np.abs(H[1:]))); axs[0, j].axhline(0, color='k', lw=0.5)
    axs[1, j].semilogx(f[1:], np.rad2deg(np.unwrap(np.angle(np.nan_to_num(H[1:])))))
    axs[0, j].set_title(s); axs[0, j].set_ylabel('|H| [dB]'); axs[1, j].set_ylabel('fase [deg]'); axs[1, j].set_xlabel('f [Hz]')
    for a in axs[:, j]: a.grid(alpha=0.3, which='both')
plt.tight_layout(); plt.show()

C:\Users\andre\AppData\Local\Temp\ipykernel_28612\2404356702.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
